# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the **FAIR² Mental Health Survey Dataset** using the `mlcroissant` library and the [Croissant schema standard](https://mlcommons.org/croissant/). We focus on record sets, fields, and transformations using strict `@id` referencing, in line with dataset FAIRness and reproducibility principles.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

----

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
We begin by loading the dataset metadata and preparing for exploration. The Croissant schema describes available record sets and fields, which aids dynamic exploration and ensures references by stable `@id` identifiers.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant dataset (metadata plus access to data)
dataset = mlc.Dataset(croissant_url)

# Access descriptive metadata (note: only access as attributes, not by key/index)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview

Explore available record sets and their associated fields. All references are made using their `@id`s for traceability.

We will print all `@id`s for record sets, then for each, display the field `@id`s and metadata.

In [ ]:
print("Available record sets (by @id, name):\n")
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"@id: {rs.id} | name: {getattr(rs, 'name', None)}")
    if hasattr(rs, 'fields'):
        print("  Fields (by @id, name):")
        for field in rs.fields:
            print(f"    - @id: {field.id}\t| name: {getattr(field, 'name', None)}")
    print("")

## 3. Data Extraction

We will load all records for each record set using their `@id`. Each record set becomes a DataFrame, with columns referencing field `@id`s whenever possible.

**Instructions:**  
- Record set and field `@id`s are used for referencing and extraction, making this notebook robust against schema changes in order or naming.
- The following code loads all record sets and stores as DataFrames for further analysis.

In [ ]:
# Extract data for each available record set into a DataFrame using @id as keys
dataframes = {}
for rs in dataset.metadata.record_sets:
    record_set_id = rs.id  # This is the @id
    print(f"Loading records for record set: {record_set_id} ...")
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df
    print(f"  - Columns: {list(df.columns)}")
    print(f"  - First 3 rows:\n{df.head(3)}\n")    # We load all record sets, but for most analyses you may pick one

## 4. Exploratory Data Analysis (EDA)

Let's select the main survey record set (usually the one with most fields such as `cr:RecordSet/survey` or similar; adjust below as needed based on the printed overview). 

Below, we demonstrate filtering, normalization, outlier handling, and simple group-by, always referencing columns by their `@id` fields.

In [ ]:
# -- Select the main record set for analysis --
# Replace with the actual main record set @id for the core survey data
main_record_set_id = None
for rs in dataset.metadata.record_sets:
    if 'survey' in rs.id or 'main' in getattr(rs, 'name', '').lower():
        main_record_set_id = rs.id
        break
if main_record_set_id is None:
    # Fall back to first record set if not found
    main_record_set_id = dataset.metadata.record_sets[0].id

# Show columns for the chosen record set
df = dataframes[main_record_set_id]
print(f"Columns in DataFrame for record set {main_record_set_id}:\n{df.columns.tolist()}")

# If the GAD-7 or PHQ-9 score field is present, use it; otherwise pick a numeric field
numeric_field_id = None
possible_scores = ['gad_7_score', 'phq_9_score', 'psq_score']
for col in df.columns:
    if any(s in col.lower() for s in possible_scores):
        numeric_field_id = col
        break
# Fallback: pick first numeric column, based on dtype
if numeric_field_id is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
print(f"Selected numeric field for EDA: {numeric_field_id}")
# Filter for values > threshold
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):\n", filtered_df.head())
# Normalize
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:\n", filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a demographic attribute, e.g., gender or age group, if present
group_field_id = None
possible_groups = ['gender', 'sex', 'age_group', 'village', 'marital_status']
for col in df.columns:
    if any(g in col.lower() for g in possible_groups):
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count','mean','std'])
    print(f"Group-by result by {group_field_id}:\n", grouped_df.head())

## 5. Visualization

We will plot distribution of the chosen numeric field (e.g., GAD-7 or PHQ-9 score) and if possible, plot grouped means by gender or another demographic variable. Visualizations use the DataFrame with columns referenced by their `@id` or logical field names.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of main numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Grouped boxplot by a demographic attribute if present
if group_field_id:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

- The `mlcroissant` library enables robust, schema-driven access to rich datasets via Croissant (`@id` + metadata).
- Using strict `@id` references protects analysis from name drift and evolution in the dataset schema.
- We explored common EDA routines: filtering, normalization, group-by, and plotting on survey data.
- The FAIR² Mental Health Survey Dataset provides a rich basis for reproducible, transparent research with clear field selection for further machine learning, statistical, or policy analysis.

**Next steps:** Explore additional record sets or link supplementary documentation using the schema's relationships and provenance metadata.